# 💼 Job Salary Prediction
### Machine Learning Project — Random Forest Regressor

**Goal**: Predict annual salary based on job title, experience, education, skills, industry, and more.

**Dataset**: 250,000 records with 9 features → 1 target (salary)

---

## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Run this cell first on Google Colab
!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Libraries imported successfully')

## 📂 Step 2 — Load Dataset

In [ ]:
# ── Option A: Upload CSV manually in Colab ──
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

# ── Option B: Read from GitHub repo (after uploading) ──
# df = pd.read_csv('https://raw.githubusercontent.com/YOUR_USERNAME/job-salary-prediction/main/data/job_salary_prediction_dataset.csv')

# ── Option C: Local path ──
df = pd.read_csv('data/job_salary_prediction_dataset.csv')

print(f'Shape: {df.shape}')
df.head()

## 🔍 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
print(df.info())
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Summary Statistics ===')
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Salary distribution
axes[0,0].hist(df['salary'], bins=50, color='#4f46e5', edgecolor='white', alpha=0.8)
axes[0,0].set_title('Salary Distribution', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Salary (USD)')
axes[0,0].set_ylabel('Count')

# Avg salary by job title
job_sal = df.groupby('job_title')['salary'].mean().sort_values(ascending=True)
job_sal.plot(kind='barh', ax=axes[0,1], color='#7c3aed', alpha=0.8)
axes[0,1].set_title('Avg Salary by Job Title', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('Avg Salary (USD)')

# Experience vs Salary
axes[1,0].scatter(df['experience_years'], df['salary'], alpha=0.05, color='#0ea5e9', s=5)
axes[1,0].set_title('Experience vs Salary', fontsize=13, fontweight='bold')
axes[1,0].set_xlabel('Experience (Years)')
axes[1,0].set_ylabel('Salary')

# Avg salary by education
edu_sal = df.groupby('education_level')['salary'].mean().sort_values()
edu_sal.plot(kind='bar', ax=axes[1,1], color='#10b981', alpha=0.8, rot=30)
axes[1,1].set_title('Avg Salary by Education', fontsize=13, fontweight='bold')
axes[1,1].set_ylabel('Avg Salary (USD)')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 EDA plots saved as eda_plots.png')

## 🔧 Step 4 — Preprocessing

In [ ]:
CATEGORICAL_COLS = ['job_title', 'education_level', 'industry',
                    'company_size', 'location', 'remote_work']
FEATURE_COLS = ['job_title', 'experience_years', 'education_level',
                'skills_count', 'industry', 'company_size',
                'location', 'remote_work', 'certifications']

label_encoders = {}
df_enc = df.copy()

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    label_encoders[col] = le
    print(f'Encoded {col}: {le.classes_.tolist()}')

X = df_enc[FEATURE_COLS]
y = df_enc['salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'\nTrain: {len(X_train):,} | Test: {len(X_test):,}')

## 🚀 Step 5 — Train Random Forest Model

In [ ]:
print('Training Random Forest Regressor...')
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)
model.fit(X_train, y_train)
print('✅ Training complete!')

## 📈 Step 6 — Evaluate Model

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('=== Model Performance ===')
print(f'MAE  : ${mae:,.0f}')
print(f'RMSE : ${rmse:,.0f}')
print(f'R²   : {r2:.4f}')

# Actual vs Predicted plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test[:3000], y_pred[:3000], alpha=0.3, color='#4f46e5', s=10)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_xlabel('Actual Salary')
ax.set_ylabel('Predicted Salary')
ax.set_title(f'Actual vs Predicted Salary (R² = {r2:.4f})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='#7c3aed', alpha=0.85)
ax.set_title('Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 Step 7 — Save Model

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/salary_model.pkl')
joblib.dump(label_encoders, 'models/label_encoders.pkl')
print('💾 Model saved to models/salary_model.pkl')
print('💾 Encoders saved to models/label_encoders.pkl')

## 🔮 Step 8 — Make a Prediction

In [ ]:
# Example prediction
sample = {
    'job_title': 'Data Scientist',
    'experience_years': 7,
    'education_level': "Master's",
    'skills_count': 12,
    'industry': 'Technology',
    'company_size': 'Large',
    'location': 'San Francisco',
    'remote_work': 'Hybrid',
    'certifications': 2
}

features = []
for feat in FEATURE_COLS:
    val = sample[feat]
    if feat in CATEGORICAL_COLS:
        val = label_encoders[feat].transform([val])[0]
    features.append(float(val))

pred_salary = model.predict([features])[0]
print(f'🔮 Predicted Salary: ${pred_salary:,.0f}')
print(f'\nFor: {sample["job_title"]} | {sample["experience_years"]} yrs | {sample["education_level"]} | {sample["location"]}')

---
## ✅ Summary

| Step | Action |
|------|--------|
| 1 | Imported libraries |
| 2 | Loaded dataset (250,000 rows) |
| 3 | Performed EDA & visualization |
| 4 | Encoded categorical features |
| 5 | Trained Random Forest Regressor |
| 6 | Evaluated: MAE, RMSE, R² |
| 7 | Saved model to `models/` folder |
| 8 | Made sample prediction |

**Next Steps**: Deploy the Flask app in `webapp/` for a browser-based salary predictor!